# Lab | Agents and Tools
## A research assistant for the bank's product team

The bank from last week is planning its 2027 products, and the product team spends hours answering the same kind of questions: *How many customers are on Premium? What does the competition charge? How many complaints were about card delivery?* Some answers are in the bank's internal documents, some are on the web, and many need a quick calculation.

Your job is to build a **research assistant**: an agent that searches the internal documents, searches the web with Tavily, and does arithmetic with a calculator, and that **cites** where each fact came from. Then you **evaluate** it: does it choose the right tools, and what does each answer cost?

**The data** is `data/bank_documents.csv`: ten short internal documents of the bank (invented for this course), with the columns `id`, `title` and `text`.

**This lab practises** building and checking an agent. You will use, from today's **Agents and Tools** lesson: tools with `@tool`, `create_agent`, the message trace and its token counts, Tavily web search, a retrieval tool over Chroma, memory with a checkpointer, and step limits.

**By the end, you can:**

1. Implement an agent with three tools (a function of your own, a retrieval tool and web search) and read its trace.
2. Evaluate the agent's tool choices and cost on a test set, and explain its failures.

Each challenge ends with a **checkpoint** that tells you what you should see. Agents make their own choices, so your traces will not match the checkpoints exactly. **The whole lab costs about one US cent and fewer than 20 Tavily searches**, well within the free plan.

## Setup

You need `pip install openai python-dotenv langchain langchain-openai langchain-tavily sentence-transformers chromadb tiktoken`. Create a `.env` file in the `your-code` folder with your two keys, as in the lesson:

```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...
```

The repository's `.gitignore` already lists `.env`: **never write a key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

The cell below loads the libraries and the documents, and gives you two helpers. `run(agent, question)` runs an agent and returns a dictionary with the answer, the tools it called, the number of model calls and the tokens. `cost(input_tokens, output_tokens)` turns tokens into US dollars.

In [1]:
import os
import logging
import warnings

os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import pandas as pd
import tiktoken
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver
from sentence_transformers import SentenceTransformer
import chromadb
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")
pd.set_option("display.max_colwidth", 100)

load_dotenv()
print("OpenAI key found:", "OPENAI_API_KEY" in os.environ, "| Tavily key found:", "TAVILY_API_KEY" in os.environ)
MODEL = "openai:gpt-5.4-nano"
PRICES = (0.20, 1.25)                        # US dollars per million input / output tokens (October 2026)

def cost(input_tokens, output_tokens):
    return (input_tokens * PRICES[0] + output_tokens * PRICES[1]) / 1_000_000

def run(agent, question, config=None):
    """Run an agent on one question and summarise what it did."""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config or {"recursion_limit": 12})
    ai_messages = [m for m in result["messages"] if type(m).__name__ == "AIMessage"]
    tokens_in = sum(m.usage_metadata["input_tokens"] for m in ai_messages)
    tokens_out = sum(m.usage_metadata["output_tokens"] for m in ai_messages)
    return {"answer": result["messages"][-1].content,
            "tools": [call["name"] for m in ai_messages for call in m.tool_calls],
            "model_calls": len(ai_messages), "input_tokens": tokens_in, "output_tokens": tokens_out,
            "cost": cost(tokens_in, tokens_out)}

documents = pd.read_csv("../data/bank_documents.csv")
documents

OpenAI key found: True | Tavily key found: True


,id,title,text
0,premium-1,Premium plan,The Premium plan costs 9.99 euros per month. It includes card payments in foreign currencies wit...
1,customers-1,Customer base,"In September 2026 the bank had 182,000 active customers, of whom 14% were on the Premium plan. 6..."
2,complaints-1,"Complaints, third quarter of 2026","The bank received 1,240 complaints between July and September 2026: 38% about card delivery, 22%..."
3,app-1,Mobile app,In September 2026 the mobile app was rated 4.3 out of 5 in the app stores. The features customer...
4,roadmap-1,Product roadmap 2027,"Planned for 2027: an AI assistant called Nova in the app in the first quarter, budgeting tools i..."
5,fx-1,Foreign currency,Card payments in a foreign currency use the Mastercard exchange rate plus a 1.5% fee. The fee is...
6,cards-2,Card delivery,"New cards arrive within 7 working days. If a card has not arrived after 10 working days, the cus..."
7,cards-3,Blocked PIN,After three wrong PIN attempts the card is blocked. The customer can unblock it in the app under...
8,transfers-1,Transfer limits,"Customers can send up to 10,000 euros per day by SEPA transfer. Instant transfers are limited to..."
9,ai-1,AI assistant policy,"The support chatbot must start every conversation with: 'You are chatting with Nova, the bank's ..."


**Checkpoint:** both keys found, and a table of 10 documents, from `premium-1` to `ai-1`.

## Challenge 1: your first tool

1. Write a tool `calculator(expression: str) -> str` with the `@tool` decorator. Its docstring must tell the model what it does. For safety, it must **refuse** any expression that contains a character other than digits, spaces and `+ - * / ( ) .` (return an error message instead), and otherwise return the result of `eval(expression)`.
2. Test it directly with `calculator.invoke(...)` on `"182000 * 0.14"` and on `"__import__('os').getcwd()"`.
3. Create an agent with `create_agent`, the model `MODEL`, the calculator as its only tool and a short system prompt. Use `run` to ask it *What is 14% of 182,000, and what is that times 9.99?* and print the result dictionary.

In [2]:
# 1. The calculator tool
ALLOWED = set("0123456789 +-*/().")

@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression and return the result, for example '182000 * 0.14' or '(1240 * 0.38) / 3'.
    Only digits, spaces and + - * / ( ) . are allowed, so write percentages as decimals (14% -> 0.14).
    Use it for every calculation instead of doing the maths yourself."""
    if not expression.strip() or not set(expression) <= ALLOWED:
        return "Error: the expression may only contain digits, spaces and + - * / ( ) ."
    if "**" in expression:                     # 9**9**9 passes the character check but would freeze the kernel
        return "Error: powers (**) are not allowed."
    try:
        return str(round(eval(expression), 10))   # round away float noise like 25480.000000000004
    except Exception as error:                 # e.g. division by zero or unbalanced brackets
        return f"Error: could not compute {expression!r} ({type(error).__name__})."

# 2. Test it directly
print(calculator.invoke("182000 * 0.14"))
print(calculator.invoke("__import__('os').getcwd()"))

# 3. A first agent
calculator_agent = create_agent(
    MODEL,
    tools=[calculator],
    system_prompt="You are a helpful assistant. Use the calculator for every calculation and answer in one or two sentences.",
)
run(calculator_agent, "What is 14% of 182,000, and what is that times 9.99?")

25480.0
Error: the expression may only contain digits, spaces and + - * / ( ) .


{'answer': '14% of 182,000 is **25,480**, and **25,480 × 9.99 = 254,545.2**.',
 'tools': ['calculator', 'calculator'],
 'model_calls': 2,
 'input_tokens': 563,
 'output_tokens': 97,
 'cost': 0.00023385000000000002}

**Checkpoint:** `25480.0`, then the error message for the second expression: the tool refused to run Python code. The agent calls the calculator twice (14% of 182,000, then that times 9.99) and answers **25,480** and **254,545.2**, with three model calls, about 600 input tokens and a cost of about $0.0002.

## Challenge 2: a tool for the bank's documents

1. Load the MiniLM embedding model (`sentence-transformers/all-MiniLM-L6-v2`), embed `title. text` for every document (normalised), and store them in a Chroma collection with cosine distance and the document `id`s as ids.
2. Write a tool `search_documents(query: str) -> str` that returns the **two** closest documents, each on one line as `[id] title: text`. Its docstring must say what is in the documents, so that the agent knows when to use it.
3. Test it directly with the query *how many customers do we have*.

In [3]:
# 1. Index the documents
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
texts = (documents["title"] + ". " + documents["text"]).tolist()
embeddings = embedder.encode(texts, normalize_embeddings=True)

chroma = chromadb.Client()
collection = chroma.get_or_create_collection("bank_documents", configuration={"hnsw": {"space": "cosine"}})
collection.upsert(                             # upsert, so re-running the cell does not fail on existing ids
    ids=documents["id"].tolist(),
    documents=documents["text"].tolist(),
    embeddings=embeddings,
    metadatas=[{"title": title} for title in documents["title"]],
)
print(collection.count(), "documents in the collection")

# 2. The retrieval tool
@tool
def search_documents(query: str) -> str:
    """Search the bank's internal documents and return the two closest ones with their id.
    They hold the bank's own facts: the Premium plan (price and what it includes), the number of customers and the
    share on Premium, the complaints of the last quarter by topic, the mobile app, the 2027 product roadmap,
    foreign currency fees, card delivery, blocked PINs, transfer limits and the policy for the AI assistant Nova.
    Use it for every question about our bank, our products, our customers or our numbers."""
    query_embedding = embedder.encode([query], normalize_embeddings=True)
    result = collection.query(query_embeddings=query_embedding, n_results=2)
    return "\n".join(f"[{doc_id}] {meta['title']}: {text}"
                     for doc_id, meta, text in zip(result["ids"][0], result["metadatas"][0], result["documents"][0]))

# 3. Test it
print(search_documents.invoke("how many customers do we have"))

10 documents in the collection
[customers-1] Customer base: In September 2026 the bank had 182,000 active customers, of whom 14% were on the Premium plan. 61% of customers are aged 25 to 44.
[complaints-1] Complaints, third quarter of 2026: The bank received 1,240 complaints between July and September 2026: 38% about card delivery, 22% about blocked cards, 15% about transfers and 25% about other topics.


**Checkpoint:** two lines, `[customers-1]` first (182,000 customers, 14% on Premium) and then the next closest document, `[complaints-1]` in the reference run.

## Challenge 3: add the web, and keep the results short

1. Create a Tavily search tool with `TavilySearch(max_results=3)`. Call it once directly with the query *Revolut Premium plan price per month Spain*, and count the **tokens** of the result with `tiktoken` (use `tiktoken.get_encoding("o200k_base")` and `len(encoding.encode(str(result)))`).
2. Long tool results are expensive, and every one is sent back to the model at each later step. Write a tool `web_search(query: str) -> str` that calls the Tavily tool, keeps only the `title`, `url` and `content` of each result, and **trims** the whole text to at most 500 tokens.
3. Create the research agent with `calculator`, `search_documents` and `web_search`. Its system prompt must say when to use each tool, ask it to cite the document id or the URL of every fact, and to answer in at most four sentences.
4. Use `run` on these three questions, and print the answer, the tools and the cost of each:
   - *How much monthly revenue does the Premium plan bring?*
   - *How does our Premium plan compare with Revolut's Premium plan in price?*
   - *Name two European fintech startups that raised money for AI products in 2026, with a source for each.*

In [4]:
# 1. Tavily, and the size of one result
tavily = TavilySearch(max_results=3)
encoding = tiktoken.get_encoding("o200k_base")

raw_result = tavily.invoke("Revolut Premium plan price per month Spain")
print("Raw Tavily result:", len(encoding.encode(str(raw_result))), "tokens")
print("Keys:", list(raw_result.keys()))

# 2. A trimmed web search tool
MAX_TOKENS = 500

def shorten(raw, max_tokens=MAX_TOKENS):
    """Keep title, url and content of every result, and cut the whole text at max_tokens."""
    text = "\n\n".join(f"{r['title']}\n{r['url']}\n{r['content']}" for r in raw.get("results", []))
    return encoding.decode(encoding.encode(text)[:max_tokens])

@tool
def web_search(query: str) -> str:
    """Search the web for public information that is NOT in the bank's internal documents: competitors and their
    prices, news, the European Central Bank, laws such as the EU AI Act, other companies and startups.
    Returns the title, URL and a short extract of the top three results."""
    try:
        raw = tavily.invoke({"query": query})
    except Exception as error:                 # no results or the API is down: tell the agent instead of crashing
        return f"Web search failed: {error}"
    return shorten(raw) if isinstance(raw, dict) else str(raw)[:2000]

print("Trimmed:", len(encoding.encode(shorten(raw_result))), "tokens")   # same result, no second search

# 3. The research agent
SYSTEM_PROMPT = """You are the research assistant of a bank's product team.
Tools:
- search_documents: the bank's own facts (products and prices, customers, complaints, app, roadmap, fees, policies). Use it first for every question about our bank.
- web_search: public information that is not in our documents (competitors, news, laws, other companies).
- calculator: every calculation, even a simple percentage. Never do arithmetic yourself.
Rules:
- Cite every fact: the document id in square brackets, e.g. [premium-1], or the URL of the web page.
- If the sources do not answer the question, say so instead of guessing.
- Answer in at most four sentences."""

research_agent = create_agent(MODEL, tools=[calculator, search_documents, web_search], system_prompt=SYSTEM_PROMPT)

# 4. Three questions
QUESTIONS = [
    "How much monthly revenue does the Premium plan bring?",
    "How does our Premium plan compare with Revolut's Premium plan in price?",
    "Name two European fintech startups that raised money for AI products in 2026, with a source for each.",
]
for question in QUESTIONS:
    result = run(research_agent, question)
    print(question)
    print(result["answer"])
    print(f"Tools: {result['tools']} | model calls: {result['model_calls']} | cost: ${result['cost']:.5f}\n")

Raw Tavily result: 968 tokens
Keys: ['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id']
Trimmed: 500 tokens


How much monthly revenue does the Premium plan bring?
Premium monthly revenue is **€254,545.20**, calculated as **182,000 active customers × 14% on Premium × €9.99/month** using [customers-1] and [premium-1].
Tools: ['search_documents', 'calculator'] | model calls: 3 | cost: $0.00051



How does our Premium plan compare with Revolut's Premium plan in price?
Our Premium plan costs **€9.99/month** [premium-1]. Revolut’s Premium plan costs **£7.99/month** [https://delphina.money/compare/revolut-plans] (also shown on Revolut’s UK pricing page as £7.99/month) [https://www.revolut.com/our-pricing-plans]. Note this is not directly comparable without converting currencies and using a specific exchange rate/date.
Tools: ['search_documents', 'web_search'] | model calls: 2 | cost: $0.00056



Name two European fintech startups that raised money for AI products in 2026, with a source for each.
- **Integral (Berlin, Germany)** — raised **€18M (Series A) on 2026-09-16** for its **AI-run accounting, tax and payroll** offering. Source: https://projectstartups.com/pages/fintech-europe  
- **Opio (Paris, France)** — raised **€4M (Seed) on 2026-09-16** for its **AI platform** that automates **financial due diligence data collection for auditors**. Source: https://projectstartups.com/pages/fintech-europe
Tools: ['web_search'] | model calls: 2 | cost: $0.00051



**Checkpoint:** about 900 tokens for one raw Tavily result, and 500 after trimming.

- **Revenue:** `search_documents` and `calculator`, about **254,545 euros per month**, citing `premium-1` and `customers-1`.
- **Revolut:** the documents and the web, sometimes the calculator too. Read the answer carefully: in the reference runs, the price came from another country (once in US dollars, once in UK pounds, from a comparison site) instead of Spain, and the agent said so.
- **Startups:** two or three web searches. In one reference run, the agent named two companies that are **not European**; in another, it said it could not find verifiable examples. Both are typical.

Each answer costs less than a tenth of a cent.

**Question:** why does trimming the search results matter, both for the cost and for the model? What could you lose by trimming too much?

**Answer:** One raw Tavily result for the Revolut query had 968 tokens, after trimming it has 500. That alone is about half of it, but the bigger point is that the tool result stays in the list of messages: every later model call in the same run sends it again, and with memory it even stays in the thread for the next questions. So a long result is paid for several times. For the model it's also better: a lot of the raw result is stuff it doesn't need (`raw_content`, scores, image lists, request ids), and with less noise it finds the one sentence with the price more easily. The `content` extracts are short and already the relevant part of the page.

If I trim too much I lose the facts themselves. The cut is made at a token count and not at the end of a sentence, so the third result is usually cut somewhere in the middle. If the price was in that part, the agent never sees it and either searches again (which costs more than the trim saved) or answers with what it has. In the worst case the URL survives but its content doesn't, and the answer cites a page the model never really read.

## Challenge 4: evaluate the tool choices

The product team wrote eight questions and, for each, the set of tools a good answer needs.

1. Run `research_agent` on every question in `TEST_SET` with `run`, and build a DataFrame with the question, the expected tools, the tools called (as a set), whether they are **exactly** the expected set (`right_tools`), the number of model calls, the tokens and the cost.
2. Print the share of questions with the right tools, the total cost, and the average cost of the questions that used `web_search` against those that did not.
3. Read the answers of the questions with the wrong tools.

In [5]:
TEST_SET = [
    ("What does the Premium plan include?", {"search_documents"}),
    ("How many customers are on the Premium plan?", {"search_documents", "calculator"}),
    ("How many complaints were about card delivery last quarter?", {"search_documents", "calculator"}),
    ("What is 17.5% of 2,400?", {"calculator"}),
    ("What is the monthly price of Revolut's Premium plan in Spain?", {"web_search"}),
    ("What did the European Central Bank decide about interest rates at its latest meeting?", {"web_search"}),
    ("When will our app get an AI assistant, and what must it tell users under the EU AI Act?", {"search_documents", "web_search"}),
    ("Do Premium customers pay a fee on card payments abroad, and how much would a standard customer pay on 300 euros?", {"search_documents", "calculator"}),
]

# 1. Run the test set
rows = []
for question, expected in TEST_SET:
    result = run(research_agent, question)
    rows.append({
        "question": question,
        "expected": expected,
        "called": set(result["tools"]),
        "right_tools": set(result["tools"]) == expected,
        "model_calls": result["model_calls"],
        "input_tokens": result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "cost": result["cost"],
        "answer": result["answer"],
    })
evaluation = pd.DataFrame(rows)
display(evaluation.drop(columns="answer"))

# 2. Right tools and cost
uses_web = evaluation["called"].apply(lambda tools: "web_search" in tools)
print(f"Right tools: {evaluation['right_tools'].sum()} of {len(evaluation)} ({evaluation['right_tools'].mean():.0%})")
print(f"Total cost: ${evaluation['cost'].sum():.4f}")
print(f"Average cost with web_search:    ${evaluation.loc[uses_web, 'cost'].mean():.5f}")
print(f"Average cost without web_search: ${evaluation.loc[~uses_web, 'cost'].mean():.5f}")

# 3. The wrong ones first; the right tools do not guarantee a right answer, so I read the others too
for row in evaluation.sort_values("right_tools").itertuples():
    label = "right tools" if row.right_tools else "WRONG TOOLS"
    print(f"\n[{label}] {row.question}\nExpected: {sorted(row.expected)} | called: {sorted(row.called)}\n{row.answer}")

,question,expected,called,right_tools,model_calls,input_tokens,output_tokens,cost
0,What does the Premium plan include?,{search_documents},{search_documents},True,2,1200,57,0.000311
1,How many customers are on the Premium plan?,"{search_documents, calculator}","{search_documents, calculator}",True,3,1908,95,0.000500
2,How many complaints were about card delivery last quarter?,"{search_documents, calculator}",{search_documents},False,2,1221,65,0.000325
3,"What is 17.5% of 2,400?",{calculator},{calculator},True,2,1126,40,0.000275
4,What is the monthly price of Revolut's Premium plan in Spain?,{web_search},{web_search},True,2,1627,65,0.000407
5,What did the European Central Bank decide about interest rates at its latest meeting?,{web_search},{web_search},True,2,1629,143,0.000505
6,"When will our app get an AI assistant, and what must it tell users under the EU AI Act?","{search_documents, web_search}","{search_documents, web_search}",True,3,2513,225,0.000784
7,"Do Premium customers pay a fee on card payments abroad, and how much would a standard customer p...","{search_documents, calculator}","{search_documents, calculator}",True,3,1951,143,0.000569


Right tools: 7 of 8 (88%)
Total cost: $0.0037
Average cost with web_search:    $0.00057
Average cost without web_search: $0.00040

[WRONG TOOLS] How many complaints were about card delivery last quarter?
Expected: ['calculator', 'search_documents'] | called: ['search_documents']
In last quarter (Q3 2026), the bank received **1,240** total complaints, of which **38%** were about **card delivery** [complaints-1].

[right tools] What does the Premium plan include?
Expected: ['search_documents'] | called: ['search_documents']
The Premium plan includes **foreign currency card payments with no fee**, **travel insurance**, and **two free ATM withdrawals abroad per month**. [premium-1]

[right tools] How many customers are on the Premium plan?
Expected: ['calculator', 'search_documents'] | called: ['calculator', 'search_documents']
As of September 2026, the bank had 182,000 active customers, and 14% were on the Premium plan [customers-1]. That corresponds to 25,480 Premium-plan customers [cust

**Checkpoint:** 7 of the 8 questions (88%) with exactly the expected tools, for a total of about a third of a cent; in the reference run the web questions cost a little more on average (about $0.0004 against $0.0003, thanks to the trimming). The miss was the card-delivery question: the agent found `complaints-1` and computed 38% of 1,240 = 471 **by itself**, without the calculator.

**Question:** look at the questions where the tools were not exactly the expected ones. Was the answer wrong, or was the test too strict? What would you change: the tool descriptions, the system prompt, or the test?

**Answer:** In my run 7 of 8 questions (88%) used exactly the expected tools. The miss was the card delivery question, like in the reference run, but in a different way: the agent found `complaints-1` and then just answered "38% of 1,240" without calculating anything. The question asks *how many*, so here the answer is really incomplete (it should be about 471), it's not only the test being strict. In my first run of the notebook, with the same prompt, it did use the calculator and got 8 of 8, so which way it goes is not stable.

Two more things I noticed when reading all the answers:

- **The right tools don't mean the right answer.** The ECB question used `web_search` as expected, but the answer is about the decision of 10 September 2026. The ECB meets about every six weeks, so in November that's probably not the latest meeting anymore, and the test would never catch it. Revolut in Spain came back as €9.99 from Revolut's Spanish page, while the comparison question in Challenge 3 ended up with the UK price (£7.99). Same fact, different wording, different source.
- **Sometimes the test is too strict.** When the model does an easy percentage correctly by itself (38% of 1,240 = 471 in the reference run), the answer is right and only the path is different.

What I would change: first the **system prompt**, it should say that a *how many* or *how much* question always needs the final number from the calculator, not just a percentage. The tool descriptions seem fine to me, the agent picked documents vs. web correctly every time. And the **test**: I would also check the answer itself (does it contain 471, does it cite `complaints-1`) and count a missing calculator call as a warning, not as a fail. For the web questions someone still has to look at the date of the source.

## Challenge 5: memory and limits

1. Create `chat_agent`: the same tools and system prompt, plus `checkpointer=InMemorySaver()`.
2. Hold this conversation in one thread (`{"configurable": {"thread_id": "product-team"}, "recursion_limit": 12}`), printing each answer:
   - *How many complaints did we get last quarter?*
   - *What share of them were about blocked cards, and how many is that?*
   - *Remind me: what was the total you started from?*
3. Ask the research agent *Compare every one of our documents with what our three main competitors offer, one by one.* with `{"recursion_limit": 4}`, and catch the `GraphRecursionError` (import it from `langgraph.errors`). What should a real application show the user when this happens?

In [6]:
from langgraph.errors import GraphRecursionError

# 1. An agent with memory
chat_agent = create_agent(MODEL, tools=[calculator, search_documents, web_search],
                          system_prompt=SYSTEM_PROMPT, checkpointer=InMemorySaver())

# 2. A conversation in one thread
THREAD = {"configurable": {"thread_id": "product-team"}, "recursion_limit": 12}
for question in ["How many complaints did we get last quarter?",
                 "What share of them were about blocked cards, and how many is that?",
                 "Remind me: what was the total you started from?"]:
    result = run(chat_agent, question, THREAD)
    # note: with a checkpointer, run() adds up the tokens of the whole thread, not only of this turn
    print(question, "\n->", result["answer"], "\n")

# 3. A step limit
try:
    result = run(research_agent, "Compare every one of our documents with what our three main competitors offer, one by one.",
                 {"recursion_limit": 4})
    print(result["answer"])
except GraphRecursionError:
    print("Stopped by the step limit")

How many complaints did we get last quarter? 
-> We received **1,240 complaints** last quarter (July–September 2026) [complaints-1]. 



What share of them were about blocked cards, and how many is that? 
-> **22%** of last quarter’s complaints were about **blocked cards** [complaints-1]. That corresponds to **272.8 complaints** out of **1,240** [complaints-1]; since complaints are whole numbers, the dataset implies **~273** blocked-card complaints. 



Remind me: what was the total you started from? 
-> The total I started from was **1,240 complaints** [complaints-1]. 



Stopped by the step limit


**What the user should see:** with `recursion_limit=4` the agent only gets about two rounds of model and tools, so a question about ten documents and three competitors can never finish. A real application shouldn't show a half-finished answer or a Python error here. It should say something like *"This question is too broad for one answer. Try one product or one competitor, for example: How does our Premium plan compare with Revolut Premium?"* and log the question, so the team can see which questions hit the limit. If that happens a lot, either the limit is too low or people need something else than a chat for it (a report, for example).

**Checkpoint:** 1,240 complaints; then 22%, about 273 (272.8); then 1,240 again, which proves the memory works. The broad question prints `Stopped by the step limit`. A real application should say that the question was too broad and suggest a narrower one (one product, one competitor), not show a half-finished answer, and it should log the case.

## What did it cost?

Add up the cost of the evaluation in Challenge 4 and compare it with the price of the team's time.

In [7]:
per_question = evaluation["cost"].mean()
print(f"Evaluation: ${evaluation['cost'].sum():.4f} for {len(evaluation)} questions, ${per_question:.5f} per question")
print(f"200 questions a day for 30 days: ${per_question * 200 * 30:.2f} per month in tokens")
print(f"Web searches in the evaluation: {uses_web.sum()} of {len(evaluation)} questions "
      f"-> about {uses_web.mean() * 200 * 30:,.0f} Tavily searches a month at 200 questions a day")

MINUTES_BY_HAND, HOURLY_COST = 5, 40              # my guess: 5 minutes per question for a person who costs 40 euros an hour
print(f"The same {len(evaluation)} questions by hand: about {len(evaluation) * MINUTES_BY_HAND / 60 * HOURLY_COST:.0f} euros of the team's time")

Evaluation: $0.0037 for 8 questions, $0.00046 per question
200 questions a day for 30 days: $2.76 per month in tokens
Web searches in the evaluation: 3 of 8 questions -> about 2,250 Tavily searches a month at 200 questions a day
The same 8 questions by hand: about 27 euros of the team's time


**Question:** the test set cost a fraction of a cent per question. If the product team asks 200 questions a day, about how much would the assistant cost per month? Which part of the cost would you watch, and what is the cost you cannot see in the token count?

**Answer:** The evaluation cost $0.0037 for 8 questions, so about $0.00046 per question. With 200 questions a day that's 6,000 a month and about **$2.76 in tokens**. That's basically nothing compared to the team's time: the same 8 questions by hand are around 27 euros if a question takes 5 minutes and a person costs 40 euros an hour (my guess, not a real number).

The part I would watch is the **web search**, not the model. 3 of the 8 questions used Tavily, so 200 questions a day are about 2,250 searches a month, and some questions search two or three times (the startup question did three searches in one of my runs). The free plan has 1,000 credits a month, after that pay-as-you-go is $0.008 per credit (tavily.com, November 2026), so roughly $10 a month more, which is already more than the tokens. Web questions also cost more tokens ($0.00057 against $0.00040 without), because the search results go back to the model. That's why the 500 token trim matters.

The cost you can't see in the token count is the **cost of a wrong answer**. A confident answer with a citation that is outdated (the ECB rate), in the wrong currency (Revolut in pounds) or about a company that isn't even a fintech ends up in a planning document and nobody notices. Someone still has to check the sources of everything that goes into a decision, and that time is the real cost. Plus building and keeping the test set up to date.

## Extra: give the assistant a voice

From the **Multimodal AI** lesson:

1. Generate a spoken question with `gpt-4o-mini-tts`, for example *How many of our customers are on the Premium plan?*, and save it as `question.mp3`.
2. Transcribe it with `gpt-4o-mini-transcribe`, send the transcript to `research_agent`, and turn the answer into speech.
3. Listen to the answer. Is a four-sentence answer with citations a good spoken answer? Change the system prompt for a voice assistant and try again.

In [8]:
from openai import OpenAI
from IPython.display import Audio, display

client = OpenAI()

def speak(text, path, instructions="Speak calmly and clearly, like a friendly bank assistant."):
    with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts", voice="alloy", input=text, instructions=instructions) as response:
        response.stream_to_file(path)

def listen(path):
    with open(path, "rb") as audio:
        return client.audio.transcriptions.create(model="gpt-4o-mini-transcribe", file=audio).text

# 1. A spoken question
speak("How many of our customers are on the Premium plan?", "question.mp3")

# 2. Transcribe, ask the agent, answer out loud
transcript = listen("question.mp3")
print("Transcript:", transcript)
result = run(research_agent, transcript)
print("Answer:", result["answer"])
speak(result["answer"], "answer.mp3")

# 3. Listen to it
display(Audio("answer.mp3"))

Transcript: How many of our customers are on the Premium plan?


Answer: As of September 2026, our bank had 182,000 active customers, and 14% were on the Premium plan [customers-1]. That equals 25,480 customers on Premium [customers-1].


In [9]:
VOICE_PROMPT = SYSTEM_PROMPT.replace(
    """- Cite every fact: the document id in square brackets, e.g. [premium-1], or the URL of the web page.
- If the sources do not answer the question, say so instead of guessing.
- Answer in at most four sentences.""",
    """- Your answer is read out loud: one or two short sentences, plain words, round numbers.
- No brackets, ids, URLs, lists or markdown. Name the source in words instead, e.g. "according to our customer report"
  or "according to Revolut's website".
- If the sources do not answer the question, say so instead of guessing.""")
assert VOICE_PROMPT != SYSTEM_PROMPT

voice_agent = create_agent(MODEL, tools=[calculator, search_documents, web_search], system_prompt=VOICE_PROMPT)
voice_result = run(voice_agent, transcript)
print("Voice answer:", voice_result["answer"])
speak(voice_result["answer"], "answer_voice.mp3")
display(Audio("answer_voice.mp3"))

Voice answer: According to our customer report, in September 2026 we had 18,200 active customers on the Premium plan.


**Is a four-sentence answer with citations a good spoken answer?** Not really. Reading it is fine, but in audio ids like `[customers-1]`, the markdown and the URLs mean nothing to the listener, and four sentences are already long to listen to. The voice prompt fixed the form: one sentence, the source in words ("according to our customer report"). But the content got worse: it said **18,200** customers on Premium, which is wrong. The real numbers are 182,000 customers in total and 25,480 on Premium. Probably the "round numbers" instruction made the model rewrite the numbers itself instead of copying them from the tools. For voice that's even worse than for text, because the listener has no citation to check. Next I would add "never change a number from the tools" to the voice prompt, and show the sources on the screen while the answer is spoken.

**Conclusions:**

- Building the agent was the easy part: three tools, a system prompt and `create_agent`. The docstrings do most of the work, the agent chose between the documents and the web correctly for every question in the test set.
- Where it failed were the small things: once it answered "38% of 1,240" instead of calculating 471, and in voice mode it turned 182,000 customers into 18,200. Both are mistakes you don't notice when you read quickly.
- The web answers are the weak spot. Right tool, but the answer depends on what Tavily returns: the UK price instead of Spain, an ECB decision that is probably not the latest one, two startups from the same aggregator page (an earlier run even named an AI venture studio that isn't a fintech). The citations make it look checked, so somebody still has to open the links.
- Memory and the step limit work: 1,240 complaints, then 22% ≈ 273, then 1,240 again from the memory, and the broad question gets stopped at 4 steps.
- Cost is not a problem: about $0.0005 per question in tokens. At the team's volume Tavily would cost more than the model.
- Next I would test the answers and not only the tools (expected numbers like 471 or 25,480), show the date of the source in web answers, and log every question that hits the step limit.